In [ ]:
from lisa import _noise_psd, _YR, _DT, sample_noise, sample_params, clean_signal
import jax.numpy as jnp
import numpy as np
from matplotlib import pyplot as plt
import jax
jax.config.update("jax_enable_x64", True)


tobs_test=_YR*1
dt_test=_DT*1
n_samples_test = int(tobs_test / dt_test)
freqs_test = jnp.fft.rfftfreq(n_samples_test, dt_test)


nt=sample_noise(t_obs=tobs_test, dt=dt_test)
nf=jnp.fft.rfft(nt)

PSD_e_try_0=_noise_psd(0,freqs_test)
PSD_e_try_2=_noise_psd(2,freqs_test)

plt.loglog(freqs_test,PSD_e_try_0)
plt.loglog(freqs_test,PSD_e_try_2)
plt.loglog(freqs_test,jnp.abs(nf[0])**2,alpha=0.2)
plt.loglog(freqs_test,jnp.abs(nf[2])**2,alpha=0.2)
plt.show()

In [ ]:
pars_test=jnp.array([1.38591126e-03, 1.32035086e-18, 1.19320621e-23, 1.03846197e-01,
       6.77106487e-01, 2.86750622e+00, 1.35587399e+00, 1.44196942e+00]) #sample_params()
print(pars_test)
gb_test=clean_signal(pars_test)

In [ ]:
sig_test=gb_test[0]
nois_test=nf[0]
dat_test=sig_test+nois_test
plt.loglog(freqs_test,jnp.abs(dat_test),alpha=0.3)
plt.loglog(freqs_test,jnp.abs(sig_test),alpha=0.3)
plt.loglog(freqs_test,jnp.abs(nois_test),alpha=0.3,c='k')
plt.xlim(pars_test[0]*0.99,pars_test[0]*1.01)
plt.ylim(1e-21,2e-18)
plt.show()

In [ ]:
source_Af, source_Ef, source_Tf = gb_test[0], gb_test[1], gb_test[2]
psd_A, psd_E, psd_T = jnp.abs(nf[0])**2, jnp.abs(nf[1])**2, jnp.abs(nf[2])**2


def matched_filter_snr_rfft(
    coeffs: np.ndarray,
    noise_psd: np.ndarray,
    freqs: np.ndarray,
    *,
    dt: float,
) -> float:
    """Matched-filter SNR using one-sided PSD and NumPy rFFT coefficients."""
    coeffs_arr = np.asarray(coeffs, dtype=np.complex128)
    noise_psd_arr = np.asarray(noise_psd, dtype=float)
    freqs_arr = np.asarray(freqs, dtype=float)
    pos = freqs_arr > 0.0
    if pos.sum() < 2:
        return 0.0
    df = float(freqs_arr[pos][1] - freqs_arr[pos][0])
    h_tilde = dt * coeffs_arr[pos]
    snr2 = 4.0 * df * np.sum(np.abs(h_tilde) ** 2 / np.maximum(noise_psd_arr[pos], 1e-60))
    return float(np.sqrt(max(float(np.real(snr2)), 0.0)))

snr_test=matched_filter_snr_rfft(dat_test,PSD_e_try_0,freqs_test,dt=dt_test)


snr_A = matched_filter_snr_rfft(source_Af, psd_A, freqs_test, dt=dt_test)
snr_E = matched_filter_snr_rfft(source_Ef, psd_E, freqs_test, dt=dt_test)
snr_T = matched_filter_snr_rfft(source_Tf, psd_T, freqs_test, dt=dt_test)
print("SNR:", np.sqrt(snr_A**2 + snr_E**2 + snr_T**2))